# HUDM 5001 — Programming for Data Science
## Session 04 · More pandas & SQLite Databases
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/humd5001_mini/blob/main/04_Pandas_SQLite.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

This notebook is used in a mini-session designed to help interested learners quickly grasp fundamental Python skills. For students of HUDM5001 in Fall 2026 semester, please look for [this repo](https://github.com/cgpan/hudm5001) for class materials.

This notebook is based on Prof. Youmi Suk's original design for Programming for Data Science. Original materials can be found [here](https://github.com/youmilab/hudm5001).

> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.

Two halves today, and they rhyme more than you would expect. First we go **deeper into pandas**: describing data honestly, cleaning up the mess, and stitching two tables together. Then we meet **SQLite**, a real database that lives in a single file — and you will notice that a `SELECT ... WHERE ... GROUP BY` is the same thought you already express with masks and `groupby`, just in another accent.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Produce a proper **descriptive summary** of a dataset, and say when the **median** beats the mean.
- Tidy a table: **rename**, **drop**, retype, and remove **duplicate** rows.
- Deal with **missing data** deliberately, rather than letting it quietly wreck your numbers.
- Summarize by group with **`groupby` + `agg`** and reshape with **`pivot_table`**.
- Combine tables with **`pd.concat`** (stacking) and **`pd.merge`** (joining on a key).
- Create a **SQLite** database, write tables into it, and query it with **SQL**.
- Read query results straight into a DataFrame with **`pd.read_sql_query`**, and translate between **SQL and pandas**.

### ✅ Prerequisites
- Sessions 01–03. Session 03's `groupby`, boolean masks, and `.loc`/`.iloc` do a lot of work today.
- Nothing to install: `sqlite3` ships with Python.

### 🧩 Key Concepts
`describe`  `agg`  `median vs mean`  `rename`  `drop`  `duplicates`  `missing data`  `pivot_table`  `concat`  `merge`  `SQLite`  `SELECT`  `WHERE`  `GROUP BY`  `JOIN`

### 📖 Reading
- McKinney, *Python for Data Analysis*, Chapters 7–8 (data cleaning, joining & reshaping)

### 🗺️ Today's Agenda
**Part 1 — More pandas**
1. Meet the data: the Titanic passenger list
2. Descriptive statistics that tell the truth
3. Tidying up: rename, drop, retype, de-duplicate
4. Missing data, handled on purpose
5. `groupby` + `agg`, and `pivot_table`
6. Combining tables: `concat` and `merge`

**Part 2 — SQLite**
7. Why a database?
8. Create a database, a table, and some rows
9. Querying with SQL: `SELECT`, `WHERE`, `ORDER BY`
10. Aggregating with `GROUP BY`
11. `JOIN` — the SQL cousin of `merge`
12. pandas ↔ SQL, side by side

---
## ▶️ Setup — run this cell first
This imports what we need and points at two small tables in the course repo: the **Titanic passenger list** and a tiny **ports** lookup table we will join to it later.

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import sqlite3          # a full database engine, built into Python — nothing to install
import json, os

# Course data, read straight from the public repo (no downloads needed).
TITANIC_URL = "https://raw.githubusercontent.com/cgpan/humd5001_mini/main/assets/titanic.csv"
PORTS_URL   = "https://raw.githubusercontent.com/cgpan/humd5001_mini/main/assets/ports.csv"

print("pandas", pd.__version__, "| sqlite3", sqlite3.sqlite_version)
print("Setup complete ✅")

---
# Part 1 — More pandas

## 2.1 Meet the data — the Titanic passenger list
891 passengers, one row each: whether they survived, their ticket class, sex, age, fare, and the port where they boarded. It is small enough to read and messy enough to be honest — there are real gaps in it, which is exactly what we want to practice on.

In [ ]:
titanic = pd.read_csv(TITANIC_URL)
print("shape:", titanic.shape)
print("columns:", list(titanic.columns))
titanic.head()

A quick note on the columns: `survived` is `1`/`0`, `pclass` is the ticket class (1st, 2nd, 3rd), `sibsp` counts siblings and spouses aboard, `parch` counts parents and children, and `embarked` is a port code (`S`, `C`, `Q`).

## 2.2 Descriptive statistics that tell the truth
`.describe()` shows you a whole numeric table at a glance. But the interesting part is what it *reveals*: look at the `fare` column below, where the mean and the 50% row (the median) are far apart.

In [ ]:
titanic[["age", "fare", "sibsp", "parch"]].describe().round(2)

The mean fare is about **32**, but the median is about **14** — and the max is over **512**. A handful of very expensive tickets drag the average upward. This is a **skewed** distribution, and it is the classic case where the **median** describes a typical passenger better than the mean does. Reporting only the mean here would quietly mislead your reader.

You can also ask for exactly the statistics you want with **`.agg()`**, passing a list of their names.

In [ ]:
# pick your own statistics instead of taking describe() as-is
print(titanic["fare"].agg(["count", "mean", "median", "std", "min", "max"]).round(2))

# and the same for several columns at once
print(titanic[["age", "fare"]].agg(["mean", "median", "std"]).round(2))

Two more everyday summaries: **`value_counts()`** for categories (add `normalize=True` to get proportions instead of counts), and **`.corr()`** for how numeric columns move together.

In [ ]:
# how many passengers in each ticket class, as counts and as shares
print(titanic["pclass"].value_counts())
print(titanic["pclass"].value_counts(normalize=True).round(3))

**Correlation** measures how two numeric columns move together, on a scale from **-1 to +1**. A value near `+1` means they rise together, near `-1` means one rises as the other falls, and near `0` means no straight-line relationship. Handing `.corr()` a DataFrame gives a whole grid — every column against every other — so the diagonal is always `1.00` (a column is perfectly correlated with itself).

In [ ]:
# the whole grid: every numeric column against every other
print(titanic[["age", "fare", "pclass", "survived"]].corr().round(2))

Read the `fare`/`pclass` cell: about **-0.55**, a solid negative relationship. That makes sense — `pclass` 1 is the *most* expensive class, so as the class number goes up, the fare goes down.

If you only care about **one pair**, call `.corr()` on a single column and pass the other. That returns one number instead of a grid, which is usually what you want in a sentence.

In [ ]:
# just one pair -> a single number
print("fare vs pclass:", round(titanic["fare"].corr(titanic["pclass"]), 3))
print("age vs fare   :", round(titanic["age"].corr(titanic["fare"]), 3))

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★★☆☆☆  **Skills:** `.mean()`, `.median()`, `.agg()`

For the `age` column: print its **mean** and its **median**, then print `count`, `min`, and `max` in one call with `.agg()`. Are the mean and median close together here? What does that tell you about the shape of the age distribution compared with `fare`?

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print("mean  :", round(titanic["age"].mean(), 2))
print("median:", titanic["age"].median())
print(titanic["age"].agg(["count", "min", "max"]))
# mean ~29.7 and median 28.0 sit close together, so age is only mildly skewed --
# unlike fare, where the mean is more than twice the median.
```

</details>

## 2.3 Tidying up: rename, drop, retype, de-duplicate
Real tables arrive with awkward names, columns you do not need, and the occasional repeated row. Four tools handle nearly all of it. Notice that each one **returns a new table** rather than editing in place, so we assign the result to a variable.

In [ ]:
# work on a copy so the original stays intact
tidy = titanic.copy()

# 1. rename: give the cryptic columns readable names
tidy = tidy.rename(columns={"sibsp": "n_siblings_spouses", "parch": "n_parents_children"})
print("renamed:", list(tidy.columns))

# 2. drop: remove a column you do not need (axis=1 means columns)
slim = tidy.drop(columns=["n_siblings_spouses", "n_parents_children"])
print("after drop:", list(slim.columns))

In [ ]:
# 3. retype: survived is really a category, not a number to average
print("before:", titanic["survived"].dtype)
as_text = titanic["survived"].astype(str)
print("after :", as_text.dtype)

# 4. duplicates: how many exact repeat rows are there?
print("duplicate rows:", titanic.duplicated().sum())
no_dupes = titanic.drop_duplicates()
print("rows before:", titanic.shape[0], "| after:", no_dupes.shape[0])

This particular passenger list has no exact duplicates, which is good news — but `duplicated().sum()` is worth running on any dataset before you trust a count.

## 2.4 Missing data, handled on purpose
You met `NaN` briefly last week. Now let us take it seriously, because how you handle gaps changes your answers. Start by finding out where they are.

In [ ]:
# how many missing values in each column?
print(titanic.isna().sum())

So `age` is missing for 177 passengers and `embarked` for 2. You have three defensible options, and the right one depends on your question:

1. **Drop the rows** — clean, but you throw away 177 real passengers.
2. **Fill the gaps** with a sensible value, usually the **median** for a skewed column.
3. **Leave them** — pandas already skips `NaN` when computing means, so sometimes doing nothing is fine.

Watch what each choice does to the row count and the average age.

In [ ]:
# option 1: drop rows missing an age
dropped = titanic.dropna(subset=["age"])
print("drop  -> rows:", dropped.shape[0], "| mean age:", round(dropped["age"].mean(), 2))

# option 2: fill missing ages with the median age
filled = titanic.copy()
filled["age"] = filled["age"].fillna(filled["age"].median())
print("fill  -> rows:", filled.shape[0], "| mean age:", round(filled["age"].mean(), 2))

# option 3: leave it alone -- mean() ignores NaN anyway
print("leave -> rows:", titanic.shape[0], "| mean age:", round(titanic["age"].mean(), 2))

Notice the mean barely moves but the **row count** changes a lot. Filling with the median keeps all 891 passengers and nudges the average only slightly — but it also invents data, so you should say so in your write-up. There is no free lunch here; there is only being explicit about what you did.

#### ✏️ Try It Yourself — Exercise 2
**Difficulty:** ★★★☆☆  **Skills:** `.isna()`, `.dropna()`, `.fillna()`, `.shape`

Work on a copy of `titanic`. (a) Print how many values are missing in `embarked`. (b) Drop the rows where `embarked` is missing and print the new row count. (c) On a *separate* copy, fill the missing `embarked` values with `"S"` (the most common port) and confirm with `.isna().sum()` that no gaps remain.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print("missing embarked:", titanic["embarked"].isna().sum())

no_missing = titanic.dropna(subset=["embarked"])
print("rows after dropping:", no_missing.shape[0])

patched = titanic.copy()
patched["embarked"] = patched["embarked"].fillna("S")
print("missing after filling:", patched["embarked"].isna().sum())
```

</details>

## 2.5 `groupby` + `agg`, and `pivot_table`
Last week `groupby` gave us one statistic per group. Pair it with **`.agg()`** and you get several at once — which is how a real descriptive table gets built.

In [ ]:
# survival rate by class: the mean of a 0/1 column IS the proportion who survived
print(titanic.groupby("pclass")["survived"].mean().round(3))

In [ ]:
# several statistics per group, in one table
print(titanic.groupby("pclass")["fare"].agg(["count", "mean", "median", "std"]).round(2))

A group summary is still a Series, so you can hand it straight back to plain Python with **`.to_dict()`** — handy when you want to drop the result into an f-string or save it as JSON.

In [ ]:
# a group summary, converted back into a Session-01 dictionary
fare_by_class = titanic.groupby("pclass")["fare"].mean().round(2).to_dict()
print(fare_by_class)
print("first class paid on average:", fare_by_class[1])

You can also group by **two** keys. Pass a list, and you get a row per combination.

In [ ]:
# survival rate for every class-and-sex combination
print(titanic.groupby(["pclass", "sex"])["survived"].mean().round(3))

That result is correct but a little cramped to read. **`pivot_table`** takes the same numbers and lays them out as a grid — one variable down the side, the other across the top. Same information, far easier on the eye.

In [ ]:
grid = pd.pivot_table(titanic,
                      values="survived",
                      index="pclass",      # rows
                      columns="sex",       # columns
                      aggfunc="mean").round(3)
print(grid)

Read that grid for a moment — it is the whole story of the disaster in six numbers. A woman in first class had roughly a 97% survival rate; a man in third class, about 14%.

> ⚠️ **One catch worth knowing:** `groupby` and `pivot_table` quietly **drop rows whose grouping key is missing**. Nobody warns you — the rows just are not there. If a grouping column has `NaN` values, check that the group totals still add up to the row count you started with.

In [ ]:
# grouping on a column with gaps loses rows -- embarked has 2 missing values
grouped_total = titanic.groupby("embarked")["survived"].count().sum()
print("rows in the table :", titanic.shape[0])
print("rows in the groups:", grouped_total, " <- the 2 passengers with no port fell out")

#### ✏️ Try It Yourself — Exercise 3
**Difficulty:** ★★★★☆  **Skills:** `groupby`, `.agg()`, `pivot_table`

(a) Group by `sex` and report `count`, `mean`, and `median` of `age` in one `.agg()` call. (b) Build a `pivot_table` of the **average `fare`**, with `pclass` down the side and `embarked` across the top, rounded to 2 decimals.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print(titanic.groupby("sex")["age"].agg(["count", "mean", "median"]).round(2))

fare_grid = pd.pivot_table(titanic,
                           values="fare",
                           index="pclass",
                           columns="embarked",
                           aggfunc="mean").round(2)
print(fare_grid)
```

</details>

## 2.6 Combining tables: `concat` and `merge`
Two different jobs, and mixing them up is a common beginner slip.

- **`pd.concat`** *stacks* tables that share the same columns — think "add more rows".
- **`pd.merge`** *joins* tables that share a **key column** — think "add more columns, matched up row by row".

In [ ]:
# concat: stack the first 3 and the last 3 passengers into one 6-row table
top = titanic.head(3)
bottom = titanic.tail(3)
stacked = pd.concat([top, bottom])
print("stacked shape:", stacked.shape)
print(stacked[["passenger_id", "sex", "age"]])

Now the interesting one. Our `embarked` column holds bare codes — `S`, `C`, `Q` — which nobody wants to read. A second little table maps each code to a real port name, and `merge` glues them together on that shared column.

In [ ]:
ports = pd.read_csv(PORTS_URL)
print(ports)

In [ ]:
# join the passenger list to the port names on the shared "embarked" column
joined = pd.merge(titanic, ports, on="embarked", how="left")
print("shape before:", titanic.shape, "-> after:", joined.shape)
print(joined[["passenger_id", "embarked", "port_name", "country"]].head())

The `how=` argument decides what happens to rows that do not find a match:

| `how=` | keeps |
|---|---|
| `"left"` | every row of the **left** table (unmatched ones get `NaN`) |
| `"right"` | every row of the **right** table |
| `"inner"` | only rows that **match in both** |
| `"outer"` | **everything** from both sides |

Remember those two passengers with a missing `embarked`? With `how="left"` they survive the join but get a `NaN` port name. With `how="inner"` they quietly vanish — and the row count tells on us.

In [ ]:
inner = pd.merge(titanic, ports, on="embarked", how="inner")
print("left join rows :", joined.shape[0])
print("inner join rows:", inner.shape[0], "  <- the 2 passengers with no port are gone")
print("missing port names after the left join:", joined["port_name"].isna().sum())

> ⚠️ **Check the row count after every join.** If it changed and you did not expect it to, something is wrong with your key. That one habit will save you hours.

#### ✏️ Try It Yourself — Exercise 4
**Difficulty:** ★★★☆☆  **Skills:** `value_counts`, `groupby`, `.mean()`

Using the `joined` table: (a) count how many passengers boarded at each **`port_name`**; (b) compute the **average fare** for each `port_name` with `groupby`, rounded to 2 decimals.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
print(joined["port_name"].value_counts())
print(joined.groupby("port_name")["fare"].mean().round(2))
```

</details>

---
# Part 2 — SQLite databases

## 2.7 Why a database?
A CSV is fine until it is not. Databases earn their keep when data gets **big** (bigger than memory), when several people need it **at once**, when the data must **survive** your script, or when it lives in **many related tables** that need to stay consistent.

**SQLite** is the gentlest possible introduction: the entire database is a single file on disk, there is no server to run, and `sqlite3` is already part of Python. It is also the most deployed database on earth — it is inside your phone, your browser, and probably the app you used this morning.

The language for talking to it is **SQL**. Here is the good news: you already think in SQL. `SELECT` is choosing columns, `WHERE` is a boolean mask, `GROUP BY` is `groupby`. Same ideas, new accent.

## 2.8 Create a database, a table, and some rows
Three steps: **connect** (this creates the file if it does not exist), get a **cursor** to run statements, and **commit** to save your changes.

We will start by hand-building a small table so you can see the machinery, then take the fast route with pandas.

In [ ]:
# connect -- creates course.db in this Colab session if it is not there yet
conn = sqlite3.connect("course.db")
cur = conn.cursor()

# start clean, so this cell can be re-run safely
cur.execute("DROP TABLE IF EXISTS holdings")

# create a table by declaring its columns and their types
cur.execute("CREATE TABLE holdings (ticker TEXT, price REAL, chg REAL)")
conn.commit()
print("table created ✅")

To insert rows, hand `executemany` a **list of tuples**. The `?` marks are placeholders that SQLite fills in safely — never paste values into a query string by hand.

In [ ]:
stocks = [
    ("NVDA", 219.00, -3.42),
    ("AAPL", 146.00, -2.73),
    ("GOOG", 2829.27, -58.20),
]

# clear the table first, so re-running this cell does not pile up duplicate rows
cur.execute("DELETE FROM holdings")

# one call inserts every row -- no loop needed
cur.executemany("INSERT INTO holdings VALUES (?, ?, ?)", stocks)
conn.commit()
print("rows inserted:", cur.rowcount)

## 2.9 Querying with SQL: `SELECT`, `WHERE`, `ORDER BY`
A query answers: *which columns* (`SELECT`), *from which table* (`FROM`), *which rows* (`WHERE`), and *in what order* (`ORDER BY`).

The friendliest way to run one is **`pd.read_sql_query`**, which hands the result straight back as a **DataFrame** — so everything you learned in Session 03 still applies.

In [ ]:
# every column, every row
pd.read_sql_query("SELECT * FROM holdings", conn)

In [ ]:
# pick columns, filter rows, and sort -- all in one statement
query = """
SELECT ticker, price
FROM holdings
WHERE price > 200
ORDER BY price DESC
"""
pd.read_sql_query(query, conn)

If you want plain Python objects instead of a DataFrame, run the query on the cursor and call **`.fetchall()`**, which returns a list of tuples.

In [ ]:
rows = cur.execute("SELECT ticker, price FROM holdings WHERE price > 200").fetchall()
print(rows)
print("first ticker:", rows[0][0])   # a list of tuples, indexed as usual

## 2.10 Aggregating with `GROUP BY`
SQL has the same summary functions you have been using: `COUNT`, `AVG`, `MIN`, `MAX`, `SUM`. Put them with `GROUP BY` and you have SQL's version of `groupby().agg()`.

Let us switch to the real data. **`to_sql`** writes a whole DataFrame into the database in one line — this is the bridge between the two halves of today's session.

In [ ]:
# write the passenger list and the ports table into the database
titanic.to_sql("passengers", conn, index=False, if_exists="replace")
ports.to_sql("ports", conn, index=False, if_exists="replace")
conn.commit()

# which tables now live in this database?
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn))

In [ ]:
# survival rate and average fare per class -- compare this with the pandas version above
query = """
SELECT pclass,
       COUNT(*)              AS n_passengers,
       ROUND(AVG(survived), 3) AS survival_rate,
       ROUND(AVG(fare), 2)     AS avg_fare
FROM passengers
GROUP BY pclass
ORDER BY pclass
"""
pd.read_sql_query(query, conn)

Line that up against `titanic.groupby("pclass")["survived"].mean()` from §2.5 — identical numbers, different accent. `AS` just renames a column in the output, the way `rename` does in pandas.

> ⚠️ **`COUNT(*)` and pandas' `.count()` are not the same thing.** `COUNT(*)` counts **rows**, including rows where a column is missing. pandas' `.count()` counts **non-missing values**. On a column with no gaps they agree; on a column with gaps they will not, and the difference is exactly the number of missing values. Neither is wrong — but if you compare a SQL summary against a pandas one, that is the first place the numbers diverge.

In [ ]:
# the same question, two ways -- watch the counts for the age column
sql_count = pd.read_sql_query("SELECT COUNT(*) AS n_rows, COUNT(age) AS n_ages FROM passengers", conn)
print(sql_count)
print("pandas .count() on age:", titanic["age"].count(), "| rows:", titanic.shape[0])
# COUNT(*) = 891 rows, but only 714 of them actually record an age.

## 2.11 `JOIN` — the SQL cousin of `merge`
Remember gluing the port names onto the passenger list with `pd.merge`? In SQL that is a **`JOIN`**, and the `ON` clause names the shared key. A `LEFT JOIN` keeps every row of the left table, exactly like `how="left"`.

In [ ]:
query = """
SELECT p.passenger_id, p.sex, p.fare, o.port_name, o.country
FROM passengers AS p
LEFT JOIN ports AS o
  ON p.embarked = o.embarked
LIMIT 5
"""
pd.read_sql_query(query, conn)

`p` and `o` are short nicknames (aliases) for the two tables, so `p.fare` reads "the `fare` column of `passengers`". `LIMIT 5` returns just the first five rows — SQL's `.head()`.

Now a join *and* an aggregate together, which is where SQL starts to feel powerful:

In [ ]:
query = """
SELECT o.port_name,
       COUNT(*)                AS n_passengers,
       ROUND(AVG(p.fare), 2)   AS avg_fare,
       ROUND(AVG(p.survived), 3) AS survival_rate
FROM passengers AS p
LEFT JOIN ports AS o
  ON p.embarked = o.embarked
GROUP BY o.port_name
ORDER BY n_passengers DESC
"""
pd.read_sql_query(query, conn)

When you are finished with a database, **close** the connection. (In a notebook you often leave it open while you work, but closing is the tidy habit.)

In [ ]:
conn.commit()
conn.close()
print("connection closed ✅")
print("the database file is still on disk:", os.path.exists("course.db"))

#### ✏️ Try It Yourself — Exercise 5
**Difficulty:** ★★★★★  **Skills:** `sqlite3.connect`, `to_sql`, `pd.read_sql_query`, `WHERE`, `GROUP BY`, `ORDER BY`

Re-open a connection to `course.db`, then answer these with **SQL** (each one query, read back with `pd.read_sql_query`):
(a) the `passenger_id`, `age`, and `fare` of the 5 passengers who paid the **highest fares**;
(b) the number of passengers and the **average age** for each `sex`;
(c) how many passengers in class 3 **survived** (`survived = 1`).

Close the connection when you are done.

In [ ]:
# Your code here


<details>
<summary>▶ Show solution</summary>

```python
conn = sqlite3.connect("course.db")

# (a) top 5 fares
q_a = """
SELECT passenger_id, age, fare
FROM passengers
ORDER BY fare DESC
LIMIT 5
"""
print(pd.read_sql_query(q_a, conn))

# (b) count and average age by sex
q_b = """
SELECT sex, COUNT(*) AS n, ROUND(AVG(age), 2) AS avg_age
FROM passengers
GROUP BY sex
"""
print(pd.read_sql_query(q_b, conn))

# (c) survivors in third class
q_c = """
SELECT COUNT(*) AS n_survivors
FROM passengers
WHERE pclass = 3 AND survived = 1
"""
print(pd.read_sql_query(q_c, conn))

conn.close()
```

</details>

## 2.12 pandas ↔ SQL, side by side
Keep this table somewhere you can find it. Once you see that the two languages express the same handful of ideas, picking up either one gets much easier.

| Task | pandas | SQL |
|---|---|---|
| all rows & columns | `df` | `SELECT * FROM t` |
| pick columns | `df[["a", "b"]]` | `SELECT a, b FROM t` |
| filter rows | `df[df["x"] > 5]` | `SELECT * FROM t WHERE x > 5` |
| two conditions | `df[(df.x > 5) & (df.y == 1)]` | `... WHERE x > 5 AND y = 1` |
| sort | `df.sort_values("x", ascending=False)` | `... ORDER BY x DESC` |
| first rows | `df.head(5)` | `... LIMIT 5` |
| count rows | `df.shape[0]` | `SELECT COUNT(*) FROM t` |
| group summary | `df.groupby("g")["x"].mean()` | `SELECT g, AVG(x) FROM t GROUP BY g` |
| join tables | `pd.merge(a, b, on="k", how="left")` | `FROM a LEFT JOIN b ON a.k = b.k` |
| rename output | `df.rename(columns={"x": "y"})` | `SELECT x AS y FROM t` |

---
## 3. Conclusions & Key Takeaways
- `.describe()` and **`.agg([...])`** build descriptive tables; when a column is **skewed**, report the **median** alongside the mean.
- Tidy with `.rename()`, `.drop(columns=...)`, `.astype()`, and `.drop_duplicates()` — each returns a **new** table.
- Find gaps with `.isna().sum()`, then **choose** deliberately: `.dropna()` loses rows, `.fillna()` invents values, doing nothing is sometimes right. Say which you did.
- **`groupby` + `.agg()`** gives several statistics per group; **`pivot_table`** lays two grouping variables out as a readable grid.
- **`concat`** stacks rows; **`merge`** joins on a key. Always check the row count after a join.
- **SQLite** is a whole database in one file, built into Python. Write tables with **`to_sql`**, read queries back with **`pd.read_sql_query`**.
- SQL's `SELECT` / `WHERE` / `GROUP BY` / `JOIN` are the same ideas as pandas' column selection, masks, `groupby`, and `merge`. Watch the one gap: `COUNT(*)` counts rows, pandas' `.count()` counts non-missing values.

> 🐍 **A gotcha for later:** statistics computed by pandas or NumPy come back as `numpy` numbers (`int64`, `float64`), and `json.dump` refuses to write those. Wrap them in `int(...)` or `float(...)` first — e.g. `int(df.shape[0])`, `float(df["x"].mean())` — and JSON will take them happily.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| custom summary | `df["x"].agg(["mean", "median", "std"])` |
| rename / drop | `df.rename(columns={...})`, `df.drop(columns=[...])` |
| duplicates | `df.duplicated().sum()`, `df.drop_duplicates()` |
| missing values | `df.isna().sum()`, `df.dropna(subset=["x"])`, `df["x"].fillna(v)` |
| group + stats | `df.groupby("g")["x"].agg(["mean", "count"])` |
| summary → dict | `df.groupby("g")["x"].mean().to_dict()` |
| correlation (pair) | `df["a"].corr(df["b"])` |
| pivot grid | `pd.pivot_table(df, values="v", index="r", columns="c", aggfunc="mean")` |
| stack / join | `pd.concat([a, b])`, `pd.merge(a, b, on="k", how="left")` |
| open a database | `conn = sqlite3.connect("my.db")` |
| DataFrame → table | `df.to_sql("t", conn, index=False, if_exists="replace")` |
| query → DataFrame | `pd.read_sql_query("SELECT * FROM t", conn)` |
| insert rows | `cur.executemany("INSERT INTO t VALUES (?, ?)", rows)` |
| save / close | `conn.commit()`, `conn.close()` |

### ⏭️ Coming Up Next
**Control structures and iterables.** At last we meet `for` and `if` — the tools we have deliberately done without. You will find that pandas already taught you *why* vectorized thinking is usually better, which makes loops much easier to use well. Read McKinney Ch 3.

### 📌 This Week's Assignment
A pandas + SQLite assignment on the **penguins** dataset — see the `assignment/` folder / GitHub.

> 💡 **Tip:** to learn SQL quickly, write the same question twice — once with pandas, once as a query — and check that the two answers agree. Do that three or four times and it will stick.